# GATOBYTE — Mini Proyecto MLOps + RAG
## Análisis de Sentimiento y Recuperación Semántica en Reseñas de Electrónica
### Notebook Exploratorio — Metodología CRISP-DM

**Equipo:** GATOBYTE | **Materia:** Machine Learning | **UCB San Pablo**

---

Este notebook sigue la metodología **CRISP-DM** aplicada al módulo RAG:

| Fase CRISP-DM | Actividad en este módulo |
|---|---|
| 1. Business Understanding | Problema de recuperación semántica |
| 2. Data Understanding | EDA del texto de reseñas |
| 3. Data Preparation | Chunking y generación de embeddings |
| 4. Modeling | Construcción del índice FAISS |
| 5. Evaluation | Precision@K, similitud coseno, latencia |
| 6. Deployment | Demo Streamlit + política de actualización |

## FASE 1 — Business Understanding

**Problema:** Los usuarios necesitan encontrar reseñas relevantes por significado,
no solo por coincidencia exacta de palabras. Una búsqueda como 
*'problemas con autonomía'* debe recuperar reseñas que hablen de *'batería que dura poco'*.

**Solución RAG:** Recuperación semántica mediante embeddings vectoriales +
índice FAISS, sobre el mismo dataset `sample_ml.parquet` usado en
las Fases 2 y 3 del proyecto.

**Restricción técnica:** Modelo preentrenado solo en **inferencia**, sin fine-tuning
ni GPU dedicada (política de la asignatura).

In [ ]:
# Instalar dependencias (solo en Colab)
# !pip install sentence-transformers faiss-cpu mlflow pyyaml streamlit -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path

# Montar Drive (Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Cargar config
with open('../config/rag_config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config cargado:')
print(f"  Modelo     : {cfg['embedding']['model_name']}")
print(f"  Chunk size : {cfg['chunking']['chunk_size']}")
print(f"  Sample size: {cfg['data']['sample_size']:,}")

## FASE 2 — Data Understanding

Exploramos las columnas de texto del parquet para entender
la distribución de longitudes y calidad del contenido.

In [ ]:
# Cargar el mismo parquet que usan las Fases 2 y 3
PARQUET_PATH = cfg['data']['parquet_path']
COLS         = cfg['data']['columns']

df = pd.read_parquet(PARQUET_PATH, columns=COLS)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# EDA: distribución de longitud de texto
df['text_len'] = df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['text_len'].clip(upper=2000), bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de longitud de reseñas (chars)')
axes[0].set_xlabel('Caracteres')

# Impacto del chunk_size elegido
chunk_size = cfg['chunking']['chunk_size']
df['num_chunks'] = (df['text_len'] / chunk_size).apply(np.ceil).astype(int)
sns.histplot(df['num_chunks'].clip(upper=10), bins=10, ax=axes[1], color='coral')
axes[1].set_title(f'Chunks por reseña (chunk_size={chunk_size})')
axes[1].set_xlabel('Número de chunks')

plt.tight_layout()
plt.savefig('../reports/eda_text_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nReseñas sin texto  : {df['text'].isna().sum():,}")
print(f"Longitud mediana   : {df['text_len'].median():.0f} chars")
print(f"Reseñas cortas (<50): {(df['text_len'] < 50).sum():,}")

In [ ]:
# EDA: distribución de sentiment (columna ya calculada en Fase 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['sentiment'].value_counts().plot(kind='bar', ax=axes[0], color=['green','gray','red'])
axes[0].set_title('Distribución de Sentimiento')
axes[0].tick_params(rotation=0)

df['main_category'].value_counts().head(6).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 6 Categorías')

plt.tight_layout()
plt.savefig('../reports/eda_sentiment_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## FASE 3 — Data Preparation

Tomamos la muestra y fragmentamos las reseñas en chunks.

In [ ]:
SAMPLE_SIZE = cfg['data']['sample_size']
CHUNK_SIZE  = cfg['chunking']['chunk_size']
MIN_LEN     = cfg['chunking']['min_chunk_length']

df_sample = (df.dropna(subset=['text'])
               .sample(min(SAMPLE_SIZE, len(df)), random_state=42)
               .reset_index(drop=True))

print(f'Muestra: {len(df_sample):,} reseñas')

# Chunking
chunks = []
for _, row in df_sample.iterrows():
    text  = str(row['text']).strip()
    parts = [text[i:i+CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
    for idx, part in enumerate(parts):
        if len(part) < MIN_LEN:
            continue
        chunks.append({
            'chunk_text'   : part,
            'rating'       : row['rating'],
            'sentiment'    : row['sentiment'],
            'main_category': row['main_category'],
        })

print(f'Total chunks: {len(chunks):,}')
print(f'Ratio chunks/reseña: {len(chunks)/len(df_sample):.2f}')

## FASE 4 — Modeling

### Transformer preentrenado (solo inferencia)

Usamos `all-MiniLM-L6-v2`, un modelo de `sentence-transformers`
optimizado para similitud semántica. Genera vectores de **384 dimensiones**.

**Cumple la política de la asignatura:**
- ✅ Solo inferencia, sin fine-tuning
- ✅ Corre en CPU sin GPU dedicada
- ✅ Modelo pequeño (~80 MB)

### Comparación vs baseline (TF-IDF)

| Aspecto | TF-IDF (baseline) | MiniLM (este proyecto) |
|---------|-------------------|------------------------|
| Representación | Frecuencia de términos | Semántica contextual |
| `"batería agota"` → `"autonomía baja"` | ❌ Sin match | ✅ Match semántico |
| Requiere GPU | ❌ No | ❌ No |
| Fine-tuning | ❌ No | ❌ No |

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, time

MODEL_NAME = cfg['embedding']['model_name']
model      = SentenceTransformer(MODEL_NAME)

texts = [c['chunk_text'] for c in chunks]
t0    = time.time()

embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # necesario para similitud coseno
)

elapsed = time.time() - t0
print(f'\nEmbeddings: {embeddings.shape}')
print(f'Tiempo    : {elapsed:.1f}s')
print(f'Velocidad : {len(texts)/elapsed:.0f} chunks/seg')

In [ ]:
# Construcción del índice FAISS
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # producto interno = coseno con vectores normalizados
index.add(embeddings.astype('float32'))

print(f'Índice FAISS: {index.ntotal:,} vectores, dim={dim}')

## FASE 5 — Evaluation

Evaluamos la calidad de recuperación con **Precision@K**.

In [ ]:
def search(query, index, chunks, model, top_k=5):
    q_vec = model.encode([query], convert_to_numpy=True,
                          normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, top_k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], indices[0]) if i >= 0]

def precision_at_k(results, keywords, k=5):
    hits = sum(1 for s, c in results[:k]
               if any(kw in c['chunk_text'].lower() for kw in keywords))
    return hits / min(k, len(results)) if results else 0.0

# Queries de evaluación del config
eval_queries = cfg['evaluation']['queries']
top_k        = cfg['evaluation']['top_k']

print(f'{'Query':<50} {'P@K':>5}  {'AvgCos':>7}')
print('-' * 65)

p_scores = []
for item in eval_queries:
    results  = search(item['query'], index, chunks, model, top_k)
    p_k      = precision_at_k(results, item['keywords'], top_k)
    avg_cos  = np.mean([s for s, _ in results])
    p_scores.append(p_k)
    print(f"{item['query'][:50]:<50} {p_k:>5.2f}  {avg_cos:>7.4f}")

print('-' * 65)
print(f"{'MEAN':<50} {np.mean(p_scores):>5.2f}")

In [ ]:
# Visualización de resultados de una query
query   = 'problemas frecuentes con el producto'
results = search(query, index, chunks, model, top_k=5)

print(f'Query: «{query}»\n')
for i, (score, chunk) in enumerate(results, 1):
    sent = chunk.get('sentiment', '')
    cat  = chunk.get('main_category', '')
    rat  = chunk.get('rating', '')
    print(f'[{i}] Score={score:.4f} | Rating={rat} | Sent={sent} | Cat={cat}')
    print(f'    {chunk["chunk_text"][:150]}...')
    print()

## FASE 6 — Deployment

El pipeline completo está implementado en los scripts `/src/`
y la demo en `/demo/app.py`.

```bash
# Construir índice (versiona automáticamente en /models/)
python src/01_build_index.py

# Evaluar calidad + guardar reporte en /reports/
python src/02_evaluate_retrieval.py

# Verificar política de actualización
python src/03_update_policy.py

# Demo interactiva
streamlit run demo/app.py

# Ver todos los runs MLflow
mlflow ui
```

### Flujo MLOps completo

```
sample_ml.parquet
      │
      ▼
[01_build_index.py]  → /models/faiss_index/v_YYYYMMDD/
      │                  ├── index.faiss
      │                  ├── chunks_metadata.pkl  
      │                  └── manifest.json
      ▼
[02_evaluate.py]     → /reports/retrieval_eval_report.json
      │                  MLflow run: eval_retrieval_quality
      ▼
[03_update_policy.py] → decisión KEEP/REBUILD
      │                  MLflow run: policy_check_*
      ▼
[demo/app.py]         → Streamlit UI
```